# 06 — GAT From First Principles
Educational notebook deriving graph attention for road-segment travel-time regression.


For node features $h_i$, a learned projection $W$, and attention vector $a$:

$e_{ij}=\mathrm{LeakyReLU}(a^T[Wh_i\Vert Wh_j])$

$lpha_{ij}=rac{\exp(e_{ij})}{\sum_{k\in\mathcal N(i)}\exp(e_{ik})}$

$h'_i=\sigma\left(\sum_{j\in\mathcal N(i)}lpha_{ij}Wh_jight)$


In [ ]:
import numpy as np

# Tiny directed segment graph: 0<-1, 0<-2, 1<-2
neighbors = {0:[1,2], 1:[2], 2:[2]}  # include self-like fallback for node 2
H = np.array([
    [0.3, 0.8],  # segment 0 state
    [0.7, 0.4],  # segment 1 state
    [0.9, 0.9],  # segment 2 state
], dtype=float)
W = np.array([[1.1, -0.2], [0.3, 0.8]])
a = np.array([0.5, -0.3, 0.7, 0.2])

def lrelu(x, alpha=0.2):
    return np.where(x > 0, x, alpha*x)


In [ ]:
Z = H @ W.T
print("Projected features Z:
", Z)

def attention_logits(i, j):
    x = np.concatenate([Z[i], Z[j]])
    return lrelu(np.dot(a, x))

alphas = {}
for i, nbrs in neighbors.items():
    logits = np.array([attention_logits(i, j) for j in nbrs])
    w = np.exp(logits - logits.max())
    w = w / w.sum()
    alphas[i] = dict(zip(nbrs, w))

alphas


In [ ]:
H_next = np.zeros_like(Z)
for i, nbrs in neighbors.items():
    out = np.zeros(Z.shape[1])
    for j in nbrs:
        out += alphas[i][j] * Z[j]
    H_next[i] = np.tanh(out)

print("Updated node representations:
", H_next)


## Interpretation for traffic segments
Each road segment updates its representation by weighting neighboring segments that can flow into it.
One snapshot cannot encode all latent causes (events outside measured features), so uncertainty remains.


In [ ]:
# multi-head illustration (2 heads) using simple perturbation
H_head1 = H_next
H_head2 = np.tanh(H_next + 0.1)
H_concat = np.concatenate([H_head1, H_head2], axis=1)
print("Concatenated multi-head shape:", H_concat.shape)
